# Graph Animation Demo

This notebook exercises the current package API with:

1. A curve draw transition with a moving pointer
2. A `fill_between` transition under that curve
3. A stress pulse that adds a temporary glow
4. A coordinated move of both the curve and the filled region
5. A final curve erase transition

The filled region remains at the end because the package does not yet include an erase-fill transition.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

plt.rcParams["animation.html"] = "jshtml"

for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Could not locate the project's src directory.")

from visualizer import (
    Curve,
    DrawTransition,
    EraseTransition,
    FillBetweenArea,
    FillBetweenTransition,
    MoveFillBetweenTransition,
    MoveTransition,
    Schedule,
    StressTransition,
)


In [ ]:
x = np.linspace(0.0, 1.0, 250)
y = np.abs(np.sin(x))
y_prime = np.square(y)

curve = Curve(
    curve_id="wave",
    x=x,
    y=y,
    color="#0f766e",
    linewidth=3.0,
)

fill = FillBetweenArea(
    fill_id="wave_fill",
    x=x,
    y1=y,
    y2=0.0,
    color="#0f766e",
    alpha=0.18,
    linestyle="--",
    linewidth=1.2,
)

schedule = (
    Schedule()
    .add(DrawTransition(curve), duration=2.0)
    .add(FillBetweenTransition(fill), duration=1.5)
    .add(StressTransition("wave", color="#14b8a6", max_alpha=0.35), duration=1.0)
    .add(MoveTransition("wave", x_prime=x, y_prime=y_prime), duration=2.0)
    .add(
        MoveFillBetweenTransition(
            "wave_fill",
            x_prime=x,
            y1_prime=y_prime,
            y2_prime=0.0,
            alpha=0.12,
        ),
        duration=2.0,
    )
    .add(StressTransition("wave", color="#f59e0b", max_alpha=0.30), duration=1.0)
    .add(EraseTransition("wave"), duration=1.5)
)

schedule.scenes()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
anim = schedule.build_animation(ax=ax, fps=30, title="Curve, Fill, Move, Stress")
ax.grid(True, alpha=0.2)
plt.close(fig)
HTML(anim.to_jshtml())
